In [2]:
# requirements_huggingface.txt
# Dependencias adicionales para Hugging Face

# Dependencias base (del archivo anterior)
mcp>=1.0.0
boto3>=1.34.0
pydantic>=2.5.0
fastapi>=0.104.0

# Hugging Face específicas
transformers>=4.36.0
torch>=2.1.0
torchvision>=0.16.0
sentence-transformers>=2.2.2
tokenizers>=0.15.0
accelerate>=0.25.0

# Para modelos especializados
datasets>=2.14.0
evaluate>=0.4.0
scikit-learn>=1.3.0

# Para HTTP requests asíncronos
httpx>=0.25.0

# Para embeddings optimizados
faiss-cpu>=1.7.4  # o faiss-gpu si tienes CUDA

# Para visualización (opcional)
matplotlib>=3.7.0
seaborn>=0.12.0

# Para procesamiento de texto médico
spacy>=3.7.0
scispacy>=0.5.3

# Para CUDA (opcional, solo si tienes GPU)
# torch>=2.1.0+cu118
# torchvision>=0.16.0+cu118
# --extra-index-url https://download.pytorch.org/whl/cu118

---
# setup_models.py - Script para descargar e inicializar modelos
#!/usr/bin/env python3
"""
Script para preparar modelos de Hugging Face para el sistema MCP médico
"""

import os
import sys
import logging
from pathlib import Path
from typing import List, Dict

import torch
from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification
from sentence_transformers import SentenceTransformer

# Configurar logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

class ModelSetup:
    """Clase para configurar y descargar modelos médicos"""
    
    def __init__(self, models_dir: str = "./models", device: str = None):
        self.models_dir = Path(models_dir)
        self.models_dir.mkdir(exist_ok=True)
        
        # Detectar dispositivo
        if device:
            self.device = device
        else:
            self.device = "cuda" if torch.cuda.is_available() else "cpu"
        
        logger.info(f"Dispositivo detectado: {self.device}")
        
        # Modelos médicos a descargar
        self.medical_models = {
            "biobert": {
                "model_name": "dmis-lab/biobert-base-cased-v1.1",
                "type": "classification",
                "description": "BioBERT para clasificación médica",
                "size_mb": 440
            },
            "clinicalbert": {
                "model_name": "emilyalsentzer/Bio_ClinicalBERT",
                "type": "classification",
                "description": "ClinicalBERT para análisis clínico",
                "size_mb": 440
            },
            "pubmedbert": {
                "model_name": "microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract",
                "type": "classification",
                "description": "PubMedBERT para literatura médica",
                "size_mb": 440
            },
            "medical_ner": {
                "model_name": "d4data/biomedical-ner-all",
                "type": "ner",
                "description": "NER especializado en entidades médicas",
                "size_mb": 440
            },
            "sentence_biobert": {
                "model_name": "pritamdeka/S-BioBert-snli-multinli-stsb",
                "type": "embeddings",
                "description": "BioBERT para embeddings de oraciones",
                "size_mb": 440
            }
        }
    
    def estimate_storage_requirements(self) -> Dict[str, float]:
        """Estima requerimientos de almacenamiento"""
        total_size = sum(model["size_mb"] for model in self.medical_models.values())
        
        return {
            "total_models_mb": total_size,
            "total_models_gb": total_size / 1024,
            "recommended_disk_gb": (total_size / 1024) * 1.5,  # 50% buffer
            "ram_requirement_gb": 8 if self.device == "cpu" else 12
        }
    
    def check_system_requirements(self) -> bool:
        """Verifica requerimientos del sistema"""
        import psutil
        
        # Verificar RAM
        ram_gb = psutil.virtual_memory().total / (1024**3)
        storage_req = self.estimate_storage_requirements()
        
        print(f"RAM disponible: {ram_gb:.1f} GB")
        print(f"RAM recomendada: {storage_req['ram_requirement_gb']} GB")
        print(f"Almacenamiento requerido: {storage_req['total_models_gb']:.1f} GB")
        
        if ram_gb < storage_req['ram_requirement_gb']:
            logger.warning("RAM insuficiente para ejecutar todos los modelos simultáneamente")
            return False
        
        return True
    
    async def download_model(self, model_key: str, force_download: bool = False) -> bool:
        """Descarga un modelo específico"""
        
        if model_key not in self.medical_models:
            logger.error(f"Modelo {model_key} no encontrado")
            return False
        
        model_info = self.medical_models[model_key]
        model_name = model_info["model_name"]
        model_type = model_info["type"]
        
        logger.info(f"Descargando {model_key}: {model_name}")
        logger.info(f"Descripción: {model_info['description']}")
        logger.info(f"Tamaño estimado: {model_info['size_mb']} MB")
        
        try:
            if model_type == "embeddings":
                # Descargar modelo de embeddings
                model = SentenceTransformer(model_name)
                model.save(str(self.models_dir / model_key))
                
            elif model_type in ["classification", "ner"]:
                # Descargar tokenizer y modelo
                tokenizer = AutoTokenizer.from_pretrained(model_name)
                
                if model_type == "classification":
                    model = AutoModelForSequenceClassification.from_pretrained(model_name)
                else:  # ner
                    model = AutoModel.from_pretrained(model_name)
                
                # Guardar localmente
                model_path = self.models_dir / model_key
                model_path.mkdir(exist_ok=True)
                
                tokenizer.save_pretrained(str(model_path))
                model.save_pretrained(str(model_path))
            
            logger.info(f"✓ Modelo {model_key} descargado exitosamente")
            return True
            
        except Exception as e:
            logger.error(f"✗ Error descargando {model_key}: {e}")
            return False
    
    async def download_all_models(self) -> Dict[str, bool]:
        """Descarga todos los modelos médicos"""
        
        print("=== Descargando Modelos Médicos de Hugging Face ===")
        storage_req = self.estimate_storage_requirements()
        
        print(f"Se descargarán {len(self.medical_models)} modelos")
        print(f"Espacio total requerido: {storage_req['total_models_gb']:.1f} GB")
        
        if not self.check_system_requirements():
            print("⚠️ Sistema no cumple requerimientos recomendados")
            response = input("¿Continuar de todos modos? (s/n): ")
            if response.lower() != 's':
                return {}
        
        results = {}
        
        for i, model_key in enumerate(self.medical_models.keys(), 1):
            print(f"\n[{i}/{len(self.medical_models)}] Procesando {model_key}...")
            success = await self.download_model(model_key)
            results[model_key] = success
            
            if success:
                print(f"✓ {model_key} completado")
            else:
                print(f"✗ {model_key} falló")
        
        return results
    
    def verify_models(self) -> Dict[str, bool]:
        """Verifica que los modelos estén correctamente descargados"""
        
        results = {}
        
        for model_key in self.medical_models.keys():
            model_path = self.models_dir / model_key
            
            if model_path.exists():
                # Verificar archivos requeridos
                required_files = ["config.json"]
                if self.medical_models[model_key]["type"] != "embeddings":
                    required_files.extend(["pytorch_model.bin", "tokenizer.json"])
                
                all_files_exist = all(
                    (model_path / file).exists() for file in required_files
                )
                results[model_key] = all_files_exist
            else:
                results[model_key] = False
        
        return results
    
    def generate_model_config(self) -> str:
        """Genera archivo de configuración para los modelos"""
        
        config = {
            "model_paths": {},
            "device": self.device,
            "models_directory": str(self.models_dir),
            "medical_models": {}
        }
        
        for model_key, model_info in self.medical_models.items():
            config["model_paths"][model_key] = str(self.models_dir / model_key)
            config["medical_models"][model_key] = {
                "type": model_info["type"],
                "description": model_info["description"],
                "original_name": model_info["model_name"]
            }
        
        import json
        return json.dumps(config, indent=2)


async def main():
    """Función principal para configurar modelos"""
    
    import argparse
    parser = argparse.ArgumentParser(description="Configurar modelos médicos de Hugging Face")
    parser.add_argument("--models-dir", default="./models", help="Directorio para modelos")
    parser.add_argument("--device", choices=["cpu", "cuda"], help="Dispositivo a usar")
    parser.add_argument("--verify-only", action="store_true", help="Solo verificar modelos existentes")
    parser.add_argument("--model", help="Descargar solo un modelo específico")
    parser.add_argument("--config-only", action="store_true", help="Solo generar configuración")
    
    args = parser.parse_args()
    
    setup = ModelSetup(models_dir=args.models_dir, device=args.device)
    
    if args.config_only:
        config = setup.generate_model_config()
        config_path = Path(args.models_dir) / "models_config.json"
        config_path.write_text(config)
        print(f"Configuración guardada en: {config_path}")
        return
    
    if args.verify_only:
        print("=== Verificando Modelos ===")
        results = setup.verify_models()
        for model_key, is_valid in results.items():
            status = "✓" if is_valid else "✗"
            print(f"{status} {model_key}: {'OK' if is_valid else 'Faltante o incompleto'}")
        return
    
    if args.model:
        # Descargar modelo específico
        success = await setup.download_model(args.model)
        if success:
            print(f"✓ Modelo {args.model} configurado exitosamente")
        else:
            print(f"✗ Error configurando modelo {args.model}")
            sys.exit(1)
    else:
        # Descargar todos los modelos
        results = await setup.download_all_models()
        
        print("\n=== Resumen de Descarga ===")
        successful = sum(results.values())
        total = len(results)
        
        for model_key, success in results.items():
            status = "✓" if success else "✗"
            print(f"{status} {model_key}")
        
        print(f"\nCompletado: {successful}/{total} modelos")
        
        if successful == total:
            # Generar archivo de configuración
            config = setup.generate_model_config()
            config_path = Path(args.models_dir) / "models_config.json"
            config_path.write_text(config)
            print(f"✓ Configuración guardada en: {config_path}")
        else:
            print("⚠️ Algunos modelos no se descargaron correctamente")
            sys.exit(1)


if __name__ == "__main__":
    import asyncio
    asyncio.run(main())

---
# docker-compose-hf.yml - Docker Compose con Hugging Face
version: '3.8'

services:
  medical-mcp-hf:
    build:
      context: .
      dockerfile: Dockerfile.huggingface
    ports:
      - "8000:8000"
    environment:
      - AWS_REGION=us-east-1
      - ENVIRONMENT=development
      - HUGGINGFACE_API_TOKEN=${HUGGINGFACE_API_TOKEN}
      - DEVICE=cpu  # Cambiar a 'cuda' si tienes GPU
      - MODELS_CACHE_DIR=/app/models
    env_file:
      - .env
    volumes:
      - ./models:/app/models:ro
      - ./logs:/var/log
      - hf_cache:/root/.cache/huggingface
    deploy:
      resources:
        limits:
          memory: 8G  # Ajustar según modelos cargados
        reservations:
          memory: 4G
    depends_on:
      - model-setup

  model-setup:
    build:
      context: .
      dockerfile: Dockerfile.model-setup
    volumes:
      - ./models:/app/models
      - hf_cache:/root/.cache/huggingface
    environment:
      - HUGGINGFACE_API_TOKEN=${HUGGINGFACE_API_TOKEN}
    command: python setup_models.py --models-dir /app/models

volumes:
  hf_cache:

---
# Dockerfile.huggingface - Dockerfile optimizado para Hugging Face
FROM python:3.11-slim

# Instalar dependencias del sistema
RUN apt-get update && apt-get install -y \
    git \
    curl \
    build-essential \
    && rm -rf /var/lib/apt/lists/*

# Crear usuario no-root
RUN groupadd -r mcpuser && useradd -r -g mcpuser mcpuser

WORKDIR /app

# Copiar requirements específicos de HF
COPY requirements_huggingface.txt .

# Instalar dependencias Python
RUN pip install --no-cache-dir -r requirements_huggingface.txt

# Instalar spaCy models para procesamiento de texto médico
RUN python -m spacy download en_core_web_sm
RUN pip install https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.3/en_core_sci_sm-0.5.3.tar.gz

# Copiar código de la aplicación
COPY . .

# Crear directorios necesarios
RUN mkdir -p /app/models /var/log && \
    chown -R mcpuser:mcpuser /app /var/log

# Cambiar a usuario no-root
USER mcpuser

# Variables de entorno para Hugging Face
ENV TRANSFORMERS_CACHE=/app/models
ENV HF_HOME=/app/models
ENV TOKENIZERS_PARALLELISM=false

# Exponer puerto
EXPOSE 8000

# Health check específico para modelos HF
HEALTHCHECK --interval=60s --timeout=10s --start-period=120s --retries=3 \
    CMD python -c "import torch; from transformers import AutoTokenizer; print('OK')" || exit 1

# Comando por defecto
CMD ["python", "huggingface_integration.py"]

---
# Dockerfile.model-setup - Contenedor para configurar modelos
FROM python:3.11-slim

RUN apt-get update && apt-get install -y \
    git \
    curl \
    && rm -rf /var/lib/apt/lists/*

WORKDIR /app

COPY requirements_huggingface.txt .
RUN pip install --no-cache-dir -r requirements_huggingface.txt

COPY setup_models.py .

# Variables de entorno
ENV TRANSFORMERS_CACHE=/app/models
ENV HF_HOME=/app/models

CMD ["python", "setup_models.py"]

---
# deploy_with_hf.sh - Script de despliegue con Hugging Face
#!/bin/bash

set -e

ENVIRONMENT=${1:-development}
AWS_REGION=${2:-us-east-1}
USE_GPU=${3:-false}

echo "Desplegando Medical MCP Server con Hugging Face"
echo "Ambiente: $ENVIRONMENT"
echo "GPU habilitado: $USE_GPU"

# Función para desplegar con modelos locales
deploy_with_local_models() {
    echo "=== Configurando Modelos Locales ==="
    
    # Verificar si existe directorio de modelos
    if [ ! -d "./models" ]; then
        echo "Creando directorio de modelos..."
        mkdir -p ./models
    fi
    
    # Descargar modelos si no existen
    if [ ! -f "./models/models_config.json" ]; then
        echo "Descargando modelos médicos de Hugging Face..."
        python setup_models.py --models-dir ./models
    else
        echo "Verificando modelos existentes..."
        python setup_models.py --models-dir ./models --verify-only
    fi
    
    echo "✓ Modelos configurados"
}

# Función para desplegar con API de Inference
deploy_with_inference_api() {
    echo "=== Configurando API de Inference ==="
    
    if [ -z "$HUGGINGFACE_API_TOKEN" ]; then
        echo "Error: HUGGINGFACE_API_TOKEN requerido para API de Inference"
        exit 1
    fi
    
    echo "✓ API Token configurado"
}

# Verificar token de Hugging Face
check_hf_token() {
    if [ -z "$HUGGINGFACE_API_TOKEN" ]; then
        echo "⚠️ HUGGINGFACE_API_TOKEN no configurado"
        echo "Opciones:"
        echo "1. export HUGGINGFACE_API_TOKEN=your_token"
        echo "2. Añadir al archivo .env"
        echo "3. Solo usar modelos pre-descargados"
        
        read -p "¿Continuar sin token? (s/n): " -n 1 -r
        echo
        if [[ ! $REPLY =~ ^[Ss]$ ]]; then
            exit 1
        fi
    fi
}

# Verificar requerimientos GPU
check_gpu_requirements() {
    if [ "$USE_GPU" = "true" ]; then
        if ! command -v nvidia-smi &> /dev/null; then
            echo "⚠️ nvidia-smi no encontrado. GPU no disponible."
            echo "Cambiando a modo CPU..."
            USE_GPU=false
        else
            echo "✓ GPU detectada:"
            nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
        fi
    fi
}

# Función principal
main() {
    echo "=== Despliegue Medical MCP + Hugging Face ==="
    
    check_hf_token
    check_gpu_requirements
    
    # Configurar variables de entorno
    export DEVICE=$([ "$USE_GPU" = "true" ] && echo "cuda" || echo "cpu")
    
    if [ "$ENVIRONMENT" = "local" ]; then
        deploy_with_local_models
        
        echo "Iniciando con Docker Compose..."
        docker-compose -f docker-compose-hf.yml up --build
        
    elif [ "$ENVIRONMENT" = "api-only" ]; then
        deploy_with_inference_api
        
        echo "Iniciando servidor con API de Inference..."
        python huggingface_integration.py --api-only
        
    else
        # Despliegue en AWS con modelos optimizados
        echo "=== Despliegue AWS con HF ==="
        
        # Optimizar para producción
        if [ "$USE_GPU" = "true" ]; then
            INSTANCE_TYPE="ml.g4dn.xlarge"
        else
            INSTANCE_TYPE="ml.m5.2xlarge"
        fi
        
        # Usar script base y añadir configuración HF
        ./deploy.sh $ENVIRONMENT $AWS_REGION
        
        # Configurar SageMaker endpoint con modelo HF personalizado
        echo "Configurando endpoint SageMaker con Hugging Face..."
        
        python - <<EOF
import boto3
from sagemaker.huggingface import HuggingFaceModel

role = "arn:aws:iam::ACCOUNT:role/SageMakerExecutionRole"

huggingface_model = HuggingFaceModel(
    transformers_version="4.36.0",
    pytorch_version="2.1.0",
    py_version="py310",
    model_data="s3://your-bucket/medical-model/",
    role=role,
    entry_point="inference.py"
)

predictor = huggingface_model.deploy(
    initial_instance_count=1,
    instance_type="$INSTANCE_TYPE",
    endpoint_name="medical-classification-hf"
)

print(f"Endpoint desplegado: {predictor.endpoint_name}")
EOF
    fi
    
    echo "✓ Despliegue completado"
}

# Función de ayuda
show_help() {
    echo "Uso: $0 [ENVIRONMENT] [AWS_REGION] [USE_GPU]"
    echo ""
    echo "ENVIRONMENT:"
    echo "  local      - Despliegue local con Docker"
    echo "  api-only   - Solo usar API de Inference de HF"
    echo "  development- Despliegue AWS de desarrollo"
    echo "  production - Despliegue AWS de producción"
    echo ""
    echo "AWS_REGION: us-east-1, us-west-2, etc."
    echo "USE_GPU: true/false"
    echo ""
    echo "Variables de entorno requeridas:"
    echo "  HUGGINGFACE_API_TOKEN - Token de Hugging Face"
    echo ""
    echo "Ejemplos:"
    echo "  $0 local                    # Despliegue local"
    echo "  $0 api-only                 # Solo API"
    echo "  $0 production us-east-1 true # AWS con GPU"
}

# Verificar argumentos
if [[ "$1" == "--help" || "$1" == "-h" ]]; then
    show_help
    exit 0
fi

main

SyntaxError: invalid decimal literal (2730613210.py, line 352)

#hola 

In [ ]:
#hoal 